In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)

# ============================================================
# 1. LOAD MODEL-READY DATA
# ============================================================

DATA_PATH = "nhl_modeling_dataset_allbooks_rolling_modelready.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
df["game_date"] = pd.to_datetime(df["game_date"])

print("Dataset shape:", df.shape)
print("Date range:", df["game_date"].min(), "->", df["game_date"].max())

# ============================================================
# 2. DEFINE TARGET AND FEATURE SET
# ============================================================

# Target: home team win
if "home_win" not in df.columns:
    df["home_win"] = (df["home_goals"] > df["away_goals"]).astype(int)

y = df["home_win"].values

# Features:
#   - rolling features: *_roll*, *_ewm*, trend_*
#   - opponent-adjusted: diff_*
#   - non-Pinnacle odds & implied probs as market features
feature_cols = []

for c in df.columns:
    if any(tag in c for tag in ["_roll", "_ewm", "trend_"]):
        feature_cols.append(c)
    elif c.startswith("diff_"):
        feature_cols.append(c)
    elif (
        ("_ml_" in c or "_spread_" in c or "_total_" in c or c.endswith("_prob"))
        and ("pinnacle" not in c)  # exclude Pinnacle from features
    ):
        feature_cols.append(c)

feature_cols = sorted(set(feature_cols) - {"home_win"})

print("Number of features:", len(feature_cols))

X = df[feature_cols].copy()

# Fill NaNs with column medians
X = X.apply(lambda col: col.fillna(col.median()))

# ============================================================
# 3. TIME-BASED TRAIN / VAL / TEST SPLIT (QUANTILES)
# ============================================================

# Sort by date to avoid any weirdness
df = df.sort_values("game_date").reset_index(drop=True)
X = X.loc[df.index]
y = y[df.index]

# Compute date cutoffs by quantiles
q_train_end = 0.6   # first 60% of games
q_val_end   = 0.8   # next 20% (60–80%)

train_cut_date = df["game_date"].quantile(q_train_end)
val_cut_date   = df["game_date"].quantile(q_val_end)

print("\nTrain/Val/Test cut dates:")
print("Train end:", train_cut_date)
print("Val end:  ", val_cut_date)

mask_train = df["game_date"] <= train_cut_date
mask_val   = (df["game_date"] > train_cut_date) & (df["game_date"] <= val_cut_date)
mask_test  = df["game_date"] > val_cut_date

X_train, y_train = X[mask_train], y[mask_train]
X_val,   y_val   = X[mask_val],   y[mask_val]
X_test,  y_test  = X[mask_test],  y[mask_test]

print("\nSplit sizes:")
print("Train:", X_train.shape[0])
print("Val:  ", X_val.shape[0])
print("Test: ", X_test.shape[0])

if X_train.shape[0] == 0 or X_val.shape[0] == 0 or X_test.shape[0] == 0:
    raise ValueError("One of the splits has 0 rows. Check date distribution / quantiles.")

# ============================================================
# 4. DEFINE MODELS
# ============================================================

log_reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    ))
])

gb_clf = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=21
)

# ============================================================
# 5. TRAIN MODELS
# ============================================================

print("\nTraining Logistic Regression...")
log_reg_pipe.fit(X_train, y_train)

print("Training Gradient Boosting...")
gb_clf.fit(X_train, y_train)

# ============================================================
# 6. EVALUATION FUNCTION
# ============================================================

def eval_model(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    print(f"\n===== {name} PERFORMANCE =====")
    for split_name, X_split, y_split in [
        ("Train", X_tr, y_tr),
        ("Val",   X_v, y_v),
        ("Test",  X_te, y_te),
    ]:
        proba = model.predict_proba(X_split)[:, 1]
        proba = np.clip(proba, 1e-6, 1 - 1e-6)
        brier = brier_score_loss(y_split, proba)
        ll    = log_loss(y_split, proba)
        auc   = roc_auc_score(y_split, proba)

        print(f"[{split_name}] Brier: {brier:.4f} | LogLoss: {ll:.4f} | ROC AUC: {auc:.4f}")

# ============================================================
# 7. EVALUATE MODELS
# ============================================================

eval_model(
    "Logistic Regression",
    log_reg_pipe,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

eval_model(
    "Gradient Boosting",
    gb_clf,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

best_model = gb_clf

# ============================================================
# 8. COMPUTE PINNACLE FAIR PROBABILITIES
# ============================================================

df_bt = df.copy()  # df is already sorted

required_pinn_cols = [
    "pinnacle_ml_home_prob",
    "pinnacle_ml_away_prob",
    "pinnacle_ml_home",
    "pinnacle_ml_away"
]

for c in required_pinn_cols:
    if c not in df_bt.columns:
        raise ValueError(f"Missing Pinnacle column: {c}")

ph_raw = df_bt["pinnacle_ml_home_prob"].astype(float)
pa_raw = df_bt["pinnacle_ml_away_prob"].astype(float)

denom = ph_raw + pa_raw
df_bt["pinnacle_fair_home_prob"] = ph_raw / denom
df_bt["pinnacle_fair_away_prob"] = pa_raw / denom

print("\nMean (fair_home + fair_away):",
      (df_bt["pinnacle_fair_home_prob"] + df_bt["pinnacle_fair_away_prob"]).mean())

# ============================================================
# 9. ADD MODEL PREDICTIONS TO DF (VAL + TEST ONLY)
# ============================================================

df_bt["model_home_win_prob"] = np.nan
df_bt.loc[mask_val,  "model_home_win_prob"] = best_model.predict_proba(X_val)[:, 1]
df_bt.loc[mask_test, "model_home_win_prob"] = best_model.predict_proba(X_test)[:, 1]
df_bt["model_home_win_prob"] = df_bt["model_home_win_prob"].clip(1e-6, 1 - 1e-6)

# ============================================================
# 10. COMPUTE EDGES VS PINNACLE
# ============================================================

df_bt["edge_home"] = df_bt["model_home_win_prob"] - df_bt["pinnacle_fair_home_prob"]
df_bt["edge_away"] = (1 - df_bt["model_home_win_prob"]) - df_bt["pinnacle_fair_away_prob"]

# ============================================================
# 11. BETTING BACKTEST
# ============================================================

def american_odds_profit_mult(odds_american):
    o = float(odds_american)
    if o > 0:
        return o / 100.0
    else:
        return 100.0 / -o

def run_backtest(df_segment, label, edge_threshold=0.02):
    seg = df_segment.copy()

    seg["best_edge"] = seg[["edge_home", "edge_away"]].max(axis=1)
    seg["bet_side"] = np.where(
        (seg["edge_home"] > seg["edge_away"]) & (seg["edge_home"] > edge_threshold),
        "home",
        np.where(
            (seg["edge_away"] > seg["edge_home"]) & (seg["edge_away"] > edge_threshold),
            "away",
            "none"
        )
    )

    bets = seg[seg["bet_side"] != "none"].copy()
    num_bets = bets.shape[0]
    if num_bets == 0:
        print(f"\n[{label}] No bets placed at edge threshold {edge_threshold}")
        return

    profits = []
    for _, row in bets.iterrows():
        if row["bet_side"] == "home":
            stake = 1.0
            price = row["pinnacle_ml_home"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 1:
                profits.append(mult * stake)
            else:
                profits.append(-stake)
        elif row["bet_side"] == "away":
            stake = 1.0
            price = row["pinnacle_ml_away"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 0:
                profits.append(mult * stake)
            else:
                profits.append(-stake)

    profits = np.array(profits)
    total_profit = profits.sum()
    roi_per_bet = total_profit / num_bets
    hit_rate = (profits > 0).mean()

    print(f"\n===== BACKTEST [{label}] =====")
    print(f"Edge threshold: {edge_threshold:.3f}")
    print(f"Number of bets: {num_bets}")
    print(f"Total profit (units): {total_profit:.2f}")
    print(f"ROI per bet: {roi_per_bet:.4f}")
    print(f"Hit rate: {hit_rate:.3f}")
    print(f"Avg edge_home on bets: {bets.loc[bets['bet_side']=='home', 'edge_home'].mean():.4f}")
    print(f"Avg edge_away on bets: {bets.loc[bets['bet_side']=='away', 'edge_away'].mean():.4f}")

# Run backtest on VAL and TEST
df_val  = df_bt[mask_val].copy()
df_test = df_bt[mask_test].copy()

run_backtest(df_val,  label="VAL",  edge_threshold=0.02)
run_backtest(df_test, label="TEST", edge_threshold=0.02)

print("\nPipeline complete.")


Dataset shape: (2951, 329)
Date range: 2020-08-03 00:00:00 -> 2024-06-24 00:00:00
Number of features: 223

Train/Val/Test cut dates:
Train end: 2023-01-12 00:00:00
Val end:   2023-11-24 00:00:00

Split sizes:
Train: 1779
Val:   586
Test:  586

Training Logistic Regression...
Training Gradient Boosting...

===== Logistic Regression PERFORMANCE =====
[Train] Brier: 0.2184 | LogLoss: 0.6249 | ROC AUC: 0.7045
[Val] Brier: 0.2389 | LogLoss: 0.6717 | ROC AUC: 0.6388
[Test] Brier: 0.2457 | LogLoss: 0.6859 | ROC AUC: 0.6572

===== Gradient Boosting PERFORMANCE =====
[Train] Brier: 0.1453 | LogLoss: 0.4602 | ROC AUC: 0.9057
[Val] Brier: 0.2351 | LogLoss: 0.6655 | ROC AUC: 0.6560
[Test] Brier: 0.2369 | LogLoss: 0.6658 | ROC AUC: 0.6367

Mean (fair_home + fair_away): 1.0

===== BACKTEST [VAL] =====
Edge threshold: 0.020
Number of bets: 537
Total profit (units): -8.29
ROI per bet: -0.0154
Hit rate: 0.464
Avg edge_home on bets: 0.1550
Avg edge_away on bets: 0.1574

===== BACKTEST [TEST] =====
Edge 

In [6]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)

# ============================================================
# 1. LOAD MODEL-READY DATA
# ============================================================

DATA_PATH = "nhl_modeling_dataset_allbooks_rolling_modelready.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
df["game_date"] = pd.to_datetime(df["game_date"])

print("Dataset shape:", df.shape)
print("Date range:", df["game_date"].min(), "->", df["game_date"].max())

# ============================================================
# 2. DEFINE TARGET AND FEATURE SET
# ============================================================

if "home_win" not in df.columns:
    df["home_win"] = (df["home_goals"] > df["away_goals"]).astype(int)

y = df["home_win"].values

feature_cols = []

for c in df.columns:
    if any(tag in c for tag in ["_roll", "_ewm", "trend_"]):
        feature_cols.append(c)
    elif c.startswith("diff_"):
        feature_cols.append(c)
    elif (
        ("_ml_" in c or "_spread_" in c or "_total_" in c or c.endswith("_prob"))
        and ("pinnacle" not in c)
    ):
        feature_cols.append(c)

feature_cols = sorted(set(feature_cols) - {"home_win"})

print("Number of features:", len(feature_cols))

X = df[feature_cols].copy()
X = X.apply(lambda col: col.fillna(col.median()))

# ============================================================
# 3. TIME-BASED TRAIN / VAL / TEST SPLIT
# ============================================================

df = df.sort_values("game_date").reset_index(drop=True)
X = X.loc[df.index]
y = y[df.index]

q_train_end = 0.6
q_val_end   = 0.8

train_cut_date = df["game_date"].quantile(q_train_end)
val_cut_date   = df["game_date"].quantile(q_val_end)

print("\nTrain/Val/Test cut dates:")
print("Train end:", train_cut_date)
print("Val end:  ", val_cut_date)

mask_train = df["game_date"] <= train_cut_date
mask_val   = (df["game_date"] > train_cut_date) & (df["game_date"] <= val_cut_date)
mask_test  = df["game_date"] > val_cut_date

X_train, y_train = X[mask_train], y[mask_train]
X_val,   y_val   = X[mask_val],   y[mask_val]
X_test,  y_test  = X[mask_test],  y[mask_test]

print("\nSplit sizes:")
print("Train:", X_train.shape[0])
print("Val:  ", X_val.shape[0])
print("Test: ", X_test.shape[0])

if X_train.shape[0] == 0 or X_val.shape[0] == 0 or X_test.shape[0] == 0:
    raise ValueError("One of the splits has 0 rows. Check date distribution / quantiles.")

# ============================================================
# 4. DEFINE & TRAIN LOGISTIC REGRESSION MODEL
# ============================================================

log_reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    ))
])

print("\nTraining Logistic Regression...")
log_reg_pipe.fit(X_train, y_train)

# ============================================================
# 5. EVALUATE LOGISTIC REGRESSION
# ============================================================

def eval_model(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    print(f"\n===== {name} PERFORMANCE =====")
    for split_name, X_split, y_split in [
        ("Train", X_tr, y_tr),
        ("Val",   X_v, y_v),
        ("Test",  X_te, y_te),
    ]:
        proba = model.predict_proba(X_split)[:, 1]
        proba = np.clip(proba, 1e-6, 1 - 1e-6)
        brier = brier_score_loss(y_split, proba)
        ll    = log_loss(y_split, proba)
        auc   = roc_auc_score(y_split, proba)

        print(f"[{split_name}] Brier: {brier:.4f} | LogLoss: {ll:.4f} | ROC AUC: {auc:.4f}")

eval_model(
    "Logistic Regression",
    log_reg_pipe,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

best_model = log_reg_pipe

# ============================================================
# 6. PINNACLE FAIR PROBS
# ============================================================

df_bt = df.copy()

for c in ["pinnacle_ml_home_prob", "pinnacle_ml_away_prob", "pinnacle_ml_home", "pinnacle_ml_away"]:
    if c not in df_bt.columns:
        raise ValueError(f"Missing Pinnacle column: {c}")

ph_raw = df_bt["pinnacle_ml_home_prob"].astype(float)
pa_raw = df_bt["pinnacle_ml_away_prob"].astype(float)

denom = ph_raw + pa_raw
df_bt["pinnacle_fair_home_prob"] = ph_raw / denom
df_bt["pinnacle_fair_away_prob"] = pa_raw / denom

print("\nMean (fair_home + fair_away):",
      (df_bt["pinnacle_fair_home_prob"] + df_bt["pinnacle_fair_away_prob"]).mean())

# ============================================================
# 7. MODEL PREDICTIONS ON VAL + TEST
# ============================================================

df_bt["model_home_win_prob"] = np.nan
df_bt.loc[mask_val,  "model_home_win_prob"] = best_model.predict_proba(X_val)[:, 1]
df_bt.loc[mask_test, "model_home_win_prob"] = best_model.predict_proba(X_test)[:, 1]
df_bt["model_home_win_prob"] = df_bt["model_home_win_prob"].clip(1e-6, 1 - 1e-6)

# ============================================================
# 8. EDGES VS PINNACLE
# ============================================================

df_bt["edge_home"] = df_bt["model_home_win_prob"] - df_bt["pinnacle_fair_home_prob"]
df_bt["edge_away"] = (1 - df_bt["model_home_win_prob"]) - df_bt["pinnacle_fair_away_prob"]

# ============================================================
# 9. BETTING BACKTEST WITH EDGE SWEEP
# ============================================================

def american_odds_profit_mult(odds_american):
    o = float(odds_american)
    if o > 0:
        return o / 100.0
    else:
        return 100.0 / -o

def run_backtest(df_segment, label, edge_threshold):
    seg = df_segment.copy()

    seg["best_edge"] = seg[["edge_home", "edge_away"]].max(axis=1)
    seg["bet_side"] = np.where(
        (seg["edge_home"] > seg["edge_away"]) & (seg["edge_home"] > edge_threshold),
        "home",
        np.where(
            (seg["edge_away"] > seg["edge_home"]) & (seg["edge_away"] > edge_threshold),
            "away",
            "none"
        )
    )

    bets = seg[seg["bet_side"] != "none"].copy()
    num_bets = bets.shape[0]
    if num_bets == 0:
        return {
            "label": label,
            "edge_threshold": edge_threshold,
            "num_bets": 0,
            "total_profit": 0.0,
            "roi_per_bet": 0.0,
            "hit_rate": np.nan,
        }

    profits = []
    for _, row in bets.iterrows():
        if row["bet_side"] == "home":
            stake = 1.0
            price = row["pinnacle_ml_home"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 1:
                profits.append(mult * stake)
            else:
                profits.append(-stake)
        elif row["bet_side"] == "away":
            stake = 1.0
            price = row["pinnacle_ml_away"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 0:
                profits.append(mult * stake)
            else:
                profits.append(-stake)

    profits = np.array(profits)
    total_profit = profits.sum()
    roi_per_bet = total_profit / num_bets
    hit_rate = (profits > 0).mean()

    return {
        "label": label,
        "edge_threshold": edge_threshold,
        "num_bets": num_bets,
        "total_profit": total_profit,
        "roi_per_bet": roi_per_bet,
        "hit_rate": hit_rate,
    }

df_val  = df_bt[mask_val].copy()
df_test = df_bt[mask_test].copy()

edge_thresholds = [0.01, 0.02, 0.03, 0.05]

results = []

for thr in edge_thresholds:
    res_val  = run_backtest(df_val,  "VAL",  thr)
    res_test = run_backtest(df_test, "TEST", thr)
    results.append(res_val)
    results.append(res_test)

print("\n===== EDGE SWEEP RESULTS =====")
for r in results:
    print(
        f"[{r['label']}] thr={r['edge_threshold']:.3f} | "
        f"bets={r['num_bets']} | "
        f"profit={r['total_profit']:.2f} | "
        f"ROI/bet={r['roi_per_bet']:.4f} | "
        f"hit={r['hit_rate']}"
    )

print("\nPipeline complete.")


Dataset shape: (2951, 329)
Date range: 2020-08-03 00:00:00 -> 2024-06-24 00:00:00
Number of features: 223

Train/Val/Test cut dates:
Train end: 2023-01-12 00:00:00
Val end:   2023-11-24 00:00:00

Split sizes:
Train: 1779
Val:   586
Test:  586

Training Logistic Regression...

===== Logistic Regression PERFORMANCE =====
[Train] Brier: 0.2184 | LogLoss: 0.6249 | ROC AUC: 0.7045
[Val] Brier: 0.2389 | LogLoss: 0.6717 | ROC AUC: 0.6388
[Test] Brier: 0.2457 | LogLoss: 0.6859 | ROC AUC: 0.6572

Mean (fair_home + fair_away): 1.0

===== EDGE SWEEP RESULTS =====
[VAL] thr=0.010 | bets=552 | profit=13.86 | ROI/bet=0.0251 | hit=0.4692028985507246
[TEST] thr=0.010 | bets=577 | profit=-14.57 | ROI/bet=-0.0252 | hit=0.4592720970537262
[VAL] thr=0.020 | bets=514 | profit=8.38 | ROI/bet=0.0163 | hit=0.46303501945525294
[TEST] thr=0.020 | bets=565 | profit=-2.57 | ROI/bet=-0.0045 | hit=0.4690265486725664
[VAL] thr=0.030 | bets=476 | profit=12.36 | ROI/bet=0.0260 | hit=0.4642857142857143
[TEST] thr=0.030

In [16]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

from xgboost import XGBClassifier  # pip install xgboost if needed

# ============================================================
# 1. LOAD MODEL-READY DATA
# ============================================================

DATA_PATH = "nhl_modeling_dataset_allbooks_rolling_modelready.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
df["game_date"] = pd.to_datetime(df["game_date"])

print("Dataset shape:", df.shape)
print("Date range:", df["game_date"].min(), "->", df["game_date"].max())

# ============================================================
# 2. DEFINE TARGET AND FEATURES
# ============================================================

if "home_win" not in df.columns:
    df["home_win"] = (df["home_goals"] > df["away_goals"]).astype(int)

y = df["home_win"].values

feature_cols = []

# Rolling / trend / opponent-adjusted / non-Pinnacle odds & probs
for c in df.columns:
    if any(tag in c for tag in ["_roll", "_ewm", "trend_"]):
        feature_cols.append(c)
    elif c.startswith("diff_"):
        feature_cols.append(c)
    elif (
        ("_ml_" in c or "_spread_" in c or "_total_" in c or c.endswith("_prob"))
        and ("pinnacle" not in c)
    ):
        feature_cols.append(c)

feature_cols = sorted(set(feature_cols) - {"home_win"})
print("Number of features:", len(feature_cols))

X = df[feature_cols].copy()
X = X.apply(lambda col: col.fillna(col.median()))

# ============================================================
# 3. TIME-BASED TRAIN / VAL / TEST SPLIT
# ============================================================

df = df.sort_values("game_date").reset_index(drop=True)
X = X.loc[df.index]
y = y[df.index]

q_train_end = 0.6
q_val_end   = 0.8

train_cut_date = df["game_date"].quantile(q_train_end)
val_cut_date   = df["game_date"].quantile(q_val_end)

print("\nTrain/Val/Test cut dates:")
print("Train end:", train_cut_date)
print("Val end:  ", val_cut_date)

mask_train = df["game_date"] <= train_cut_date
mask_val   = (df["game_date"] > train_cut_date) & (df["game_date"] <= val_cut_date)
mask_test  = df["game_date"] > val_cut_date

X_train, y_train = X[mask_train], y[mask_train]
X_val,   y_val   = X[mask_val],   y[mask_val]
X_test,  y_test  = X[mask_test],  y[mask_test]

print("\nSplit sizes:")
print("Train:", X_train.shape[0])
print("Val:  ", X_val.shape[0])
print("Test: ", X_test.shape[0])

if X_train.shape[0] == 0 or X_val.shape[0] == 0 or X_test.shape[0] == 0:
    raise ValueError("One of the splits has 0 rows. Check date distribution / quantiles.")

# ============================================================
# 4. DEFINE MODELS: LOGISTIC (UNCAL), LOGISTIC (CALIBRATED), XGBOOST
# ============================================================

# Uncalibrated logistic regression
log_reg_uncal = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced"
    ))
])

# Base model BEFORE calibration (no calibration yet)
log_reg_base_for_calib = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced"
    ))
])

# Calibrated logistic regression (ISOTONIC)
# NOTE: estimator passed POSITIONALLY
log_reg_calibrated = CalibratedClassifierCV(
    log_reg_base_for_calib,   # positional — works in all sklearn versions
    method="isotonic",
    cv=3
)

# XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=21,
    n_jobs=-1,
    tree_method="hist"
)


# ============================================================
# 5. TRAIN MODELS
# ============================================================

print("\nTraining Logistic Regression (uncalibrated)...")
log_reg_uncal.fit(X_train, y_train)

print("Training Logistic Regression (calibrated, isotonic)...")
log_reg_calibrated.fit(X_train, y_train)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# ============================================================
# 6. EVALUATION FUNCTION
# ============================================================

def eval_model(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    print(f"\n===== {name} PERFORMANCE =====")
    for split_name, X_split, y_split in [
        ("Train", X_tr, y_tr),
        ("Val",   X_v, y_v),
        ("Test",  X_te, y_te),
    ]:
        proba = model.predict_proba(X_split)[:, 1]
        proba = np.clip(proba, 1e-6, 1 - 1e-6)
        brier = brier_score_loss(y_split, proba)
        ll    = log_loss(y_split, proba)
        auc   = roc_auc_score(y_split, proba)

        print(f"[{split_name}] Brier: {brier:.4f} | LogLoss: {ll:.4f} | ROC AUC: {auc:.4f}")

# ============================================================
# 7. COMPARE MODELS
# ============================================================

eval_model(
    "Logistic (Uncalibrated)",
    log_reg_uncal,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

eval_model(
    "Logistic (Calibrated Isotonic)",
    log_reg_calibrated,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

eval_model(
    "XGBoost",
    xgb_model,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

# Choose calibrated logistic as primary for backtesting
best_model = log_reg_calibrated

# ============================================================
# 8. PINNACLE FAIR PROBABILITIES
# ============================================================

df_bt = df.copy()

for c in ["pinnacle_ml_home_prob", "pinnacle_ml_away_prob",
          "pinnacle_ml_home", "pinnacle_ml_away"]:
    if c not in df_bt.columns:
        raise ValueError(f"Missing Pinnacle column: {c}")

ph_raw = df_bt["pinnacle_ml_home_prob"].astype(float)
pa_raw = df_bt["pinnacle_ml_away_prob"].astype(float)

denom = ph_raw + pa_raw
df_bt["pinnacle_fair_home_prob"] = ph_raw / denom
df_bt["pinnacle_fair_away_prob"] = pa_raw / denom

print("\nMean (fair_home + fair_away):",
      (df_bt["pinnacle_fair_home_prob"] + df_bt["pinnacle_fair_away_prob"]).mean())

# ============================================================
# 9. ADD MODEL PREDICTIONS (VAL + TEST) FOR BACKTEST
# ============================================================

df_bt["model_home_win_prob_calibrated"] = np.nan
df_bt.loc[mask_val,  "model_home_win_prob_calibrated"] = best_model.predict_proba(X_val)[:, 1]
df_bt.loc[mask_test, "model_home_win_prob_calibrated"] = best_model.predict_proba(X_test)[:, 1]
df_bt["model_home_win_prob_calibrated"] = df_bt["model_home_win_prob_calibrated"].clip(1e-6, 1 - 1e-6)

# Optionally also store XGBoost probs for later comparison/backtests
df_bt["model_home_win_prob_xgb"] = np.nan
df_bt.loc[mask_val,  "model_home_win_prob_xgb"] = xgb_model.predict_proba(X_val)[:, 1]
df_bt.loc[mask_test, "model_home_win_prob_xgb"] = xgb_model.predict_proba(X_test)[:, 1]
df_bt["model_home_win_prob_xgb"] = df_bt["model_home_win_prob_xgb"].clip(1e-6, 1 - 1e-6)

# ============================================================
# 10. EDGES VS PINNACLE (CALIBRATED MODEL)
# ============================================================

df_bt["edge_home_calib"] = df_bt["model_home_win_prob_calibrated"] - df_bt["pinnacle_fair_home_prob"]
df_bt["edge_away_calib"] = (1 - df_bt["model_home_win_prob_calibrated"]) - df_bt["pinnacle_fair_away_prob"]

# (Optional) edges for XGBoost
df_bt["edge_home_xgb"] = df_bt["model_home_win_prob_xgb"] - df_bt["pinnacle_fair_home_prob"]
df_bt["edge_away_xgb"] = (1 - df_bt["model_home_win_prob_xgb"]) - df_bt["pinnacle_fair_away_prob"]

# ============================================================
# 11. BETTING BACKTEST HELPERS
# ============================================================

def american_odds_profit_mult(odds_american):
    o = float(odds_american)
    if o > 0:
        return o / 100.0
    else:
        return 100.0 / -o

def run_backtest(df_segment, label, edge_threshold, edge_home_col, edge_away_col):
    seg = df_segment.copy()

    seg["best_edge"] = seg[[edge_home_col, edge_away_col]].max(axis=1)
    seg["bet_side"] = np.where(
        (seg[edge_home_col] > seg[edge_away_col]) & (seg[edge_home_col] > edge_threshold),
        "home",
        np.where(
            (seg[edge_away_col] > seg[edge_home_col]) & (seg[edge_away_col] > edge_threshold),
            "away",
            "none"
        )
    )

    bets = seg[seg["bet_side"] != "none"].copy()
    num_bets = bets.shape[0]
    if num_bets == 0:
        return {
            "label": label,
            "edge_threshold": edge_threshold,
            "num_bets": 0,
            "total_profit": 0.0,
            "roi_per_bet": 0.0,
            "hit_rate": np.nan,
        }

    profits = []
    for _, row in bets.iterrows():
        if row["bet_side"] == "home":
            stake = 1.0
            price = row["pinnacle_ml_home"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 1:
                profits.append(mult * stake)
            else:
                profits.append(-stake)
        elif row["bet_side"] == "away":
            stake = 1.0
            price = row["pinnacle_ml_away"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 0:
                profits.append(mult * stake)
            else:
                profits.append(-stake)

    profits = np.array(profits)
    total_profit = profits.sum()
    roi_per_bet = total_profit / num_bets
    hit_rate = (profits > 0).mean()

    return {
        "label": label,
        "edge_threshold": edge_threshold,
        "num_bets": num_bets,
        "total_profit": total_profit,
        "roi_per_bet": roi_per_bet,
        "hit_rate": hit_rate,
    }

# ============================================================
# 12. RUN EDGE SWEEP BACKTESTS (CALIBRATED VS XGBOOST)
# ============================================================

df_val  = df_bt[mask_val].copy()
df_test = df_bt[mask_test].copy()

edge_thresholds = [0.01, 0.02, 0.03, 0.05]

results = []

for thr in edge_thresholds:
    # Calibrated logistic
    res_val_calib  = run_backtest(df_val,  f"VAL-Calib",  thr, "edge_home_calib", "edge_away_calib")
    res_test_calib = run_backtest(df_test, f"TEST-Calib", thr, "edge_home_calib", "edge_away_calib")
    results.append(res_val_calib)
    results.append(res_test_calib)

    # XGBoost
    res_val_xgb  = run_backtest(df_val,  f"VAL-XGB",  thr, "edge_home_xgb", "edge_away_xgb")
    res_test_xgb = run_backtest(df_test, f"TEST-XGB", thr, "edge_home_xgb", "edge_away_xgb")
    results.append(res_val_xgb)
    results.append(res_test_xgb)

print("\n===== EDGE SWEEP RESULTS (CALIBRATED LOGISTIC vs XGBOOST) =====")
for r in results:
    print(
        f"[{r['label']}] thr={r['edge_threshold']:.3f} | "
        f"bets={r['num_bets']} | "
        f"profit={r['total_profit']:.2f} | "
        f"ROI/bet={r['roi_per_bet']:.4f} | "
        f"hit={r['hit_rate']}"
    )

print("\nPipeline complete.")


Dataset shape: (2951, 329)
Date range: 2020-08-03 00:00:00 -> 2024-06-24 00:00:00
Number of features: 223

Train/Val/Test cut dates:
Train end: 2023-01-12 00:00:00
Val end:   2023-11-24 00:00:00

Split sizes:
Train: 1779
Val:   586
Test:  586

Training Logistic Regression (uncalibrated)...
Training Logistic Regression (calibrated, isotonic)...
Training XGBoost...

===== Logistic (Uncalibrated) PERFORMANCE =====
[Train] Brier: 0.2184 | LogLoss: 0.6249 | ROC AUC: 0.7045
[Val] Brier: 0.2389 | LogLoss: 0.6717 | ROC AUC: 0.6388
[Test] Brier: 0.2457 | LogLoss: 0.6859 | ROC AUC: 0.6572

===== Logistic (Calibrated Isotonic) PERFORMANCE =====
[Train] Brier: 0.2171 | LogLoss: 0.6212 | ROC AUC: 0.7087
[Val] Brier: 0.2361 | LogLoss: 0.6812 | ROC AUC: 0.6357
[Test] Brier: 0.2373 | LogLoss: 0.6652 | ROC AUC: 0.6547

===== XGBoost PERFORMANCE =====
[Train] Brier: 0.1394 | LogLoss: 0.4447 | ROC AUC: 0.9143
[Val] Brier: 0.2353 | LogLoss: 0.6680 | ROC AUC: 0.6574
[Test] Brier: 0.2379 | LogLoss: 0.6691 |

In [18]:
# Assume df_bt already built from the last pipeline run
# and has: model_home_win_prob_xgb, edge_home_xgb, edge_away_xgb, etc.

def run_backtest_per_season(df_bt, mask, edge_threshold=0.02):
    seg = df_bt[mask].copy()
    seg["season"] = seg["season"].astype(str) if "season" in seg.columns else seg["game_date"].dt.year.astype(str)

    results = []

    def american_odds_profit_mult(o):
        o = float(o)
        if o > 0:
            return o / 100.0
        else:
            return 100.0 / -o

    for season, df_season in seg.groupby("season"):
        df_season = df_season.copy()
        df_season["best_edge"] = df_season[["edge_home_xgb", "edge_away_xgb"]].max(axis=1)
        df_season["bet_side"] = np.where(
            (df_season["edge_home_xgb"] > df_season["edge_away_xgb"]) & (df_season["edge_home_xgb"] > edge_threshold),
            "home",
            np.where(
                (df_season["edge_away_xgb"] > df_season["edge_home_xgb"]) & (df_season["edge_away_xgb"] > edge_threshold),
                "away",
                "none"
            )
        )

        bets = df_season[df_season["bet_side"] != "none"].copy()
        if bets.empty:
            continue

        profits = []
        for _, row in bets.iterrows():
            if row["bet_side"] == "home":
                price = row["pinnacle_ml_home"]
                mult = american_odds_profit_mult(price)
                profits.append(mult if row["home_win"] == 1 else -1.0)
            else:
                price = row["pinnacle_ml_away"]
                mult = american_odds_profit_mult(price)
                profits.append(mult if row["home_win"] == 0 else -1.0)

        profits = np.array(profits)
        results.append({
            "season": season,
            "num_bets": len(profits),
            "total_profit": profits.sum(),
            "roi_per_bet": profits.mean()
        })

    return pd.DataFrame(results)

# VAL and TEST season-wise results
val_season_results  = run_backtest_per_season(df_bt, mask_val,  edge_threshold=0.02)
test_season_results = run_backtest_per_season(df_bt, mask_test, edge_threshold=0.02)

print("VAL season-wise results:")
print(val_season_results)

print("\nTEST season-wise results:")
print(test_season_results)


VAL season-wise results:
  season  num_bets  total_profit  roi_per_bet
0   2023       410     31.213586     0.076131
1   2024       135     -9.868407    -0.073099

TEST season-wise results:
  season  num_bets  total_profit  roi_per_bet
0   2024       524     16.408375     0.031314


In [20]:
import numpy as np
import pandas as pd

# ============================================================
# CONSENSUS MODEL BACKTEST
# ============================================================

# Safety checks
required_cols = [
    "model_home_win_prob_calibrated",
    "model_home_win_prob_xgb",
    "edge_home_calib", "edge_away_calib",
    "edge_home_xgb", "edge_away_xgb",
    "home_win",
    "pinnacle_ml_home", "pinnacle_ml_away",
    "game_date"
]

for c in required_cols:
    if c not in df_bt.columns:
        raise ValueError(f"df_bt is missing required column: {c}")

def american_odds_profit_mult(o):
    o = float(o)
    if o > 0:
        return o / 100.0
    else:
        return 100.0 / -o

def run_consensus_backtest(df_segment, label, edge_threshold=0.02,
                           require_logit_edge_positive=True):
    """
    Consensus rule:
      - Both models agree on side (both >0.5 = home, both <0.5 = away)
      - XGBoost edge on that side >= edge_threshold
      - Optionally require calibrated logistic edge on that side > 0
    """
    seg = df_segment.copy()

    # Determine each model's side
    seg["logit_side"] = np.where(
        seg["model_home_win_prob_calibrated"] > 0.5, "home", "away"
    )
    seg["xgb_side"] = np.where(
        seg["model_home_win_prob_xgb"] > 0.5, "home", "away"
    )

    # Consensus side: both must match
    seg["consensus_side"] = np.where(
        seg["logit_side"] == seg["xgb_side"],
        seg["logit_side"],
        "none"
    )

    # Start with no bets
    seg["bet_side"] = "none"

    # HOME consensus bets
    home_mask = seg["consensus_side"].eq("home")
    if require_logit_edge_positive:
        home_mask = home_mask & (seg["edge_home_calib"] > 0)
    home_mask = home_mask & (seg["edge_home_xgb"] >= edge_threshold)

    seg.loc[home_mask, "bet_side"] = "home"

    # AWAY consensus bets
    away_mask = seg["consensus_side"].eq("away")
    if require_logit_edge_positive:
        away_mask = away_mask & (seg["edge_away_calib"] > 0)
    away_mask = away_mask & (seg["edge_away_xgb"] >= edge_threshold)

    seg.loc[away_mask, "bet_side"] = "away"

    # Filter to actual bets
    bets = seg[seg["bet_side"] != "none"].copy()
    num_bets = bets.shape[0]
    if num_bets == 0:
        return {
            "label": label,
            "edge_threshold": edge_threshold,
            "num_bets": 0,
            "total_profit": 0.0,
            "roi_per_bet": 0.0,
            "hit_rate": np.nan,
        }

    profits = []
    for _, row in bets.iterrows():
        stake = 1.0
        if row["bet_side"] == "home":
            price = row["pinnacle_ml_home"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 1:
                profits.append(mult * stake)
            else:
                profits.append(-stake)
        else:  # away
            price = row["pinnacle_ml_away"]
            mult = american_odds_profit_mult(price)
            if row["home_win"] == 0:
                profits.append(mult * stake)
            else:
                profits.append(-stake)

    profits = np.array(profits)
    total_profit = profits.sum()
    roi_per_bet = total_profit / num_bets
    hit_rate = (profits > 0).mean()

    return {
        "label": label,
        "edge_threshold": edge_threshold,
        "num_bets": num_bets,
        "total_profit": total_profit,
        "roi_per_bet": roi_per_bet,
        "hit_rate": hit_rate,
    }

# ------------------------------------------------------------
# RUN CONSENSUS BACKTEST (VAL + TEST) FOR MULTIPLE THRESHOLDS
# ------------------------------------------------------------

df_val  = df_bt[mask_val].copy()
df_test = df_bt[mask_test].copy()

edge_thresholds = [0.01, 0.02, 0.03, 0.05]

cons_results = []
for thr in edge_thresholds:
    res_val  = run_consensus_backtest(df_val,  f"VAL-Consensus",  edge_threshold=thr)
    res_test = run_consensus_backtest(df_test, f"TEST-Consensus", edge_threshold=thr)
    cons_results.append(res_val)
    cons_results.append(res_test)

print("\n===== CONSENSUS MODEL EDGE SWEEP RESULTS =====")
for r in cons_results:
    print(
        f"[{r['label']}] thr={r['edge_threshold']:.3f} | "
        f"bets={r['num_bets']} | "
        f"profit={r['total_profit']:.2f} | "
        f"ROI/bet={r['roi_per_bet']:.4f} | "
        f"hit={r['hit_rate']}"
    )



===== CONSENSUS MODEL EDGE SWEEP RESULTS =====
[VAL-Consensus] thr=0.010 | bets=276 | profit=4.55 | ROI/bet=0.0165 | hit=0.5
[TEST-Consensus] thr=0.010 | bets=279 | profit=15.52 | ROI/bet=0.0556 | hit=0.5340501792114696
[VAL-Consensus] thr=0.020 | bets=272 | profit=3.27 | ROI/bet=0.0120 | hit=0.4963235294117647
[TEST-Consensus] thr=0.020 | bets=275 | profit=15.93 | ROI/bet=0.0579 | hit=0.5345454545454545
[VAL-Consensus] thr=0.030 | bets=267 | profit=3.93 | ROI/bet=0.0147 | hit=0.4943820224719101
[TEST-Consensus] thr=0.030 | bets=267 | profit=13.70 | ROI/bet=0.0513 | hit=0.5280898876404494
[VAL-Consensus] thr=0.050 | bets=244 | profit=-2.47 | ROI/bet=-0.0101 | hit=0.4713114754098361
[TEST-Consensus] thr=0.050 | bets=255 | profit=11.58 | ROI/bet=0.0454 | hit=0.5215686274509804


In [22]:
# ============================================================
# SEASON-WISE CONSENSUS BACKTEST
# ============================================================

def run_consensus_backtest_per_season(df_bt, mask, edge_threshold=0.02,
                                      require_logit_edge_positive=True):
    seg = df_bt[mask].copy()

    if "season" in seg.columns:
        seg["season_str"] = seg["season"].astype(str)
    else:
        seg["season_str"] = seg["game_date"].dt.year.astype(str)

    rows = []

    for season, df_season in seg.groupby("season_str"):
        res = run_consensus_backtest(
            df_season,
            label=f"Season-{season}",
            edge_threshold=edge_threshold,
            require_logit_edge_positive=require_logit_edge_positive
        )
        rows.append({
            "season": season,
            "num_bets": res["num_bets"],
            "total_profit": res["total_profit"],
            "roi_per_bet": res["roi_per_bet"]
        })

    return pd.DataFrame(rows)

cons_val_seasons  = run_consensus_backtest_per_season(df_bt, mask_val,  edge_threshold=0.02)
cons_test_seasons = run_consensus_backtest_per_season(df_bt, mask_test, edge_threshold=0.02)

print("\nSeason-wise CONSENSUS (VAL, thr=0.02):")
print(cons_val_seasons)

print("\nSeason-wise CONSENSUS (TEST, thr=0.02):")
print(cons_test_seasons)



Season-wise CONSENSUS (VAL, thr=0.02):
  season  num_bets  total_profit  roi_per_bet
0   2023       199     15.487296     0.077826
1   2024        73    -12.214455    -0.167321

Season-wise CONSENSUS (TEST, thr=0.02):
  season  num_bets  total_profit  roi_per_bet
0   2024       275     15.933812     0.057941


In [24]:
import numpy as np
import pandas as pd

# ============================================================
# 0. COMMON HELPERS
# ============================================================

def american_odds_profit_mult(o):
    """Return profit multiple for a 1-unit stake given American odds."""
    o = float(o)
    if o > 0:
        return o / 100.0
    else:
        return 100.0 / -o

def american_to_decimal(o):
    """Convert American odds to decimal odds."""
    o = float(o)
    if o > 0:
        return 1.0 + o / 100.0
    else:
        return 1.0 + 100.0 / -o

# ============================================================
# 1. BUILD CONSENSUS BETS DATAFRAME (DETAILED PER-BET ROWS)
# ============================================================

def build_consensus_bets(
    df_bt,
    mask,
    edge_threshold=0.02,
    require_logit_edge_positive=True,
    use_xgb_edges=True
):
    """
    Build a detailed bets DataFrame under the consensus rule:
      - XGB and calibrated logistic agree on side
      - Edge (XGB or logistic) on that side >= edge_threshold
      - Optionally require calibrated logistic edge in same direction > 0
    Returns a DataFrame with one row per bet, including price, prob, edge, etc.
    """
    seg = df_bt[mask].copy()
    seg = seg.reset_index(drop=True)

    # Determine side from each model
    seg["logit_side"] = np.where(
        seg["model_home_win_prob_calibrated"] > 0.5, "home", "away"
    )
    seg["xgb_side"] = np.where(
        seg["model_home_win_prob_xgb"] > 0.5, "home", "away"
    )

    # Consensus: both must agree
    seg["consensus_side"] = np.where(
        seg["logit_side"] == seg["xgb_side"],
        seg["logit_side"],
        "none"
    )

    # Choose which edge columns to use as primary
    if use_xgb_edges:
        edge_home_col = "edge_home_xgb"
        edge_away_col = "edge_away_xgb"
    else:
        edge_home_col = "edge_home_calib"
        edge_away_col = "edge_away_calib"

    # Start with no bets
    seg["bet_side"] = "none"

    # HOME consensus bets
    home_mask = seg["consensus_side"].eq("home")
    if require_logit_edge_positive:
        home_mask = home_mask & (seg["edge_home_calib"] > 0)
    home_mask = home_mask & (seg[edge_home_col] >= edge_threshold)
    seg.loc[home_mask, "bet_side"] = "home"

    # AWAY consensus bets
    away_mask = seg["consensus_side"].eq("away")
    if require_logit_edge_positive:
        away_mask = away_mask & (seg["edge_away_calib"] > 0)
    away_mask = away_mask & (seg[edge_away_col] >= edge_threshold)
    seg.loc[away_mask, "bet_side"] = "away"

    bets = seg[seg["bet_side"] != "none"].copy()
    if bets.empty:
        return bets  # empty DataFrame

    # Attach bet-specific info
    bets["bet_price_american"] = np.where(
        bets["bet_side"] == "home",
        bets["pinnacle_ml_home"],
        bets["pinnacle_ml_away"]
    )

    # Model win prob from XGB (primary alpha source)
    bets["p_model"] = np.where(
        bets["bet_side"] == "home",
        bets["model_home_win_prob_xgb"],
        1.0 - bets["model_home_win_prob_xgb"]
    )

    # Book "fair" prob from Pinnacle (already vig-stripped)
    bets["p_fair_book"] = np.where(
        bets["bet_side"] == "home",
        bets["pinnacle_fair_home_prob"],
        bets["pinnacle_fair_away_prob"]
    )

    # Edge used for selection
    bets["edge_model_vs_book"] = np.where(
        bets["bet_side"] == "home",
        bets[edge_home_col],
        bets[edge_away_col]
    )

    # Outcome profit for a 1-unit flat stake
    profit_list = []
    for _, row in bets.iterrows():
        price = row["bet_price_american"]
        mult = american_odds_profit_mult(price)
        if row["bet_side"] == "home":
            profit_list.append(mult if row["home_win"] == 1 else -1.0)
        else:
            profit_list.append(mult if row["home_win"] == 0 else -1.0)

    bets["profit_flat_unit"] = profit_list

    # Add season + month for segmentation
    if "season" in bets.columns:
        bets["season_str"] = bets["season"].astype(str)
    else:
        bets["season_str"] = bets["game_date"].dt.year.astype(str)

    bets["month"] = bets["game_date"].dt.month

    # Favorite vs dog
    bets["is_favorite"] = bets["bet_price_american"] < 0
    bets["is_home_bet"] = bets["bet_side"] == "home"

    return bets


# Build VAL and TEST consensus bets at thr=0.02 (you can change thr)
cons_bets_val = build_consensus_bets(df_bt, mask_val,  edge_threshold=0.02)
cons_bets_test = build_consensus_bets(df_bt, mask_test, edge_threshold=0.02)

print("Consensus VAL bets:", cons_bets_val.shape[0])
print("Consensus TEST bets:", cons_bets_test.shape[0])

# ============================================================
# 2. KELLY / FRACTIONAL KELLY STAKING + BANKROLL SIMULATION
# ============================================================

def add_kelly_columns(bets_df):
    """
    Add full Kelly fraction and decimal odds columns based on
    model probability and Pinnacle price.
    """
    bets = bets_df.copy()

    # Decimal odds and b = decimal - 1
    bets["decimal_odds"] = bets["bet_price_american"].apply(american_to_decimal)
    bets["b"] = bets["decimal_odds"] - 1.0

    # Kelly fraction: f* = (b*p - q) / b, where p=model prob, q=1-p
    bets["q_model"] = 1.0 - bets["p_model"]
    bets["kelly_full"] = (bets["b"] * bets["p_model"] - bets["q_model"]) / bets["b"]

    # No negative Kelly — if edge <= 0, set Kelly to 0
    bets["kelly_full"] = bets["kelly_full"].clip(lower=0.0)

    return bets

def simulate_kelly_bankroll(
    bets_df,
    initial_bankroll=1000.0,
    frac_kelly=0.25,
    max_fraction_per_bet=0.05
):
    """
    Simulate bankroll evolution using fractional Kelly staking on consensus bets.
    - initial_bankroll: starting bankroll units
    - frac_kelly: fraction of full Kelly to use (e.g., 0.25 = quarter Kelly)
    - max_fraction_per_bet: cap stake as fraction of current bankroll
    Returns:
      - path_df: bankroll path over bets
      - summary dict
    """
    bets = add_kelly_columns(bets_df)

    bankroll = initial_bankroll
    bankroll_path = []
    max_bankroll = initial_bankroll
    max_drawdown = 0.0

    for i, row in bets.iterrows():
        k_full = row["kelly_full"]
        if k_full <= 0:
            stake_fraction = 0.0
        else:
            stake_fraction = min(frac_kelly * k_full, max_fraction_per_bet)

        stake = stake_fraction * bankroll

        # If stake is effectively zero, skip bet
        if stake < 1e-6:
            bankroll_path.append({
                "idx": i,
                "bankroll_before": bankroll,
                "stake": 0.0,
                "bankroll_after": bankroll,
                "stake_fraction": 0.0
            })
            continue

        price = row["bet_price_american"]
        mult = american_odds_profit_mult(price)

        if row["bet_side"] == "home":
            win = row["home_win"] == 1
        else:
            win = row["home_win"] == 0

        if win:
            profit = mult * stake
        else:
            profit = -stake

        new_bankroll = bankroll + profit

        # Track drawdown
        max_bankroll = max(max_bankroll, new_bankroll)
        drawdown = (max_bankroll - new_bankroll) / max_bankroll
        max_drawdown = max(max_drawdown, drawdown)

        bankroll_path.append({
            "idx": i,
            "bankroll_before": bankroll,
            "stake": stake,
            "bankroll_after": new_bankroll,
            "stake_fraction": stake_fraction,
            "profit": profit,
            "win": win
        })

        bankroll = new_bankroll

    path_df = pd.DataFrame(bankroll_path)
    total_profit = bankroll - initial_bankroll
    roi_total = total_profit / initial_bankroll

    # Count bets actually taken
    num_bets = (path_df["stake"] > 0).sum()
    hit_rate = path_df.loc[path_df["stake"] > 0, "win"].mean()

    summary = {
        "initial_bankroll": initial_bankroll,
        "final_bankroll": bankroll,
        "total_profit": total_profit,
        "roi_total": roi_total,
        "num_bets": int(num_bets),
        "hit_rate": float(hit_rate),
        "max_drawdown": float(max_drawdown),
        "frac_kelly": frac_kelly,
        "max_fraction_per_bet": max_fraction_per_bet
    }

    return path_df, summary

# Example: simulate Kelly on TEST consensus bets
kelly_path_test, kelly_summary_test = simulate_kelly_bankroll(
    cons_bets_test,
    initial_bankroll=1000.0,
    frac_kelly=0.25,
    max_fraction_per_bet=0.05
)

print("\n=== Kelly Simulation (TEST Consensus) ===")
for k, v in kelly_summary_test.items():
    print(f"{k}: {v}")

# ============================================================
# 3. LINE-RANGE ANALYSIS (ROI BY ODDS BAND)
# ============================================================

def line_range_analysis(bets_df, band_edges=None):
    """
    Analyze ROI by American odds bands.
    band_edges: list of breakpoints for pd.cut on American odds.
    """
    bets = bets_df.copy()
    if band_edges is None:
        # Example bands: huge dog, medium dog, small dog/fav, medium fav, big fav
        band_edges = [-10000, -300, -150, 0, 150, 300, 10000]

    bets["price_band"] = pd.cut(
        bets["bet_price_american"].astype(float),
        bins=band_edges,
        include_lowest=True
    )

    grouped = bets.groupby("price_band")["profit_flat_unit"].agg(
        count="count",
        total_profit="sum",
        roi_per_bet=lambda x: x.mean()
    ).reset_index()

    return grouped

line_ranges_val = line_range_analysis(cons_bets_val)
line_ranges_test = line_range_analysis(cons_bets_test)

print("\n=== Line Range Analysis (VAL, Consensus) ===")
print(line_ranges_val)

print("\n=== Line Range Analysis (TEST, Consensus) ===")
print(line_ranges_test)

# ============================================================
# 4. SEGMENTED PERFORMANCE (SEASON / MONTH / FAV-DOG / SIDE)
# ============================================================

def segmented_performance(bets_df):
    """
    Compute ROI per segment across interesting dimensions:
      - season
      - month
      - is_favorite
      - is_home_bet
    """
    bets = bets_df.copy()

    # Season-level
    season_grp = bets.groupby("season_str")["profit_flat_unit"].agg(
        count="count",
        total_profit="sum",
        roi_per_bet=lambda x: x.mean()
    ).reset_index()

    # Month-level across all seasons
    month_grp = bets.groupby("month")["profit_flat_unit"].agg(
        count="count",
        total_profit="sum",
        roi_per_bet=lambda x: x.mean()
    ).reset_index()

    # Favorite vs dog
    fav_grp = bets.groupby("is_favorite")["profit_flat_unit"].agg(
        count="count",
        total_profit="sum",
        roi_per_bet=lambda x: x.mean()
    ).reset_index()

    # Home vs away bets
    side_grp = bets.groupby("is_home_bet")["profit_flat_unit"].agg(
        count="count",
        total_profit="sum",
        roi_per_bet=lambda x: x.mean()
    ).reset_index()

    return season_grp, month_grp, fav_grp, side_grp

season_val, month_val, fav_val, side_val = segmented_performance(cons_bets_val)
season_test, month_test, fav_test, side_test = segmented_performance(cons_bets_test)

print("\n=== Segmented Performance (VAL) ===")
print("By season:")
print(season_val)
print("\nBy month:")
print(month_val)
print("\nFavorite vs Dog:")
print(fav_val)
print("\nHome vs Away bets:")
print(side_val)

print("\n=== Segmented Performance (TEST) ===")
print("By season:")
print(season_test)
print("\nBy month:")
print(month_test)
print("\nFavorite vs Dog:")
print(fav_test)
print("\nHome vs Away bets:")
print(side_test)

# ============================================================
# 5. DAILY DEPLOYMENT: GENERATE BET SHEET FOR NEW GAMES
# ============================================================

def generate_daily_bets(
    games_features_df,
    pinn_odds_df,
    calib_model,
    xgb_model,
    feature_cols,
    edge_threshold=0.02,
    require_logit_edge_positive=True,
    frac_kelly=0.25,
    max_fraction_per_bet=0.05,
    bankroll=1000.0
):
    """
    Given:
      - games_features_df: upcoming games with the SAME feature columns as training
      - pinn_odds_df: DataFrame with Pinnacle odds (ml_home, ml_away, fair probs)
      - calib_model: fitted calibrated logistic model
      - xgb_model: fitted XGBoost model
      - feature_cols: list of feature columns to use
    Returns:
      - bets_today: DataFrame of recommended bets with model probs, edges, and Kelly stakes.
    """

    df_new = games_features_df.copy()

    # Merge Pinnacle odds onto feature set (on game_id or some key)
    # Here assume both have 'game_id'
    df_new = df_new.merge(
        pinn_odds_df,
        on="game_id",
        how="inner",
        suffixes=("", "_pinn")
    )

    # Compute fair probs from Pinnacle probabilities if needed
    if not {"pinnacle_fair_home_prob", "pinnacle_fair_away_prob"}.issubset(df_new.columns):
        if {"pinnacle_ml_home_prob", "pinnacle_ml_away_prob"}.issubset(df_new.columns):
            ph_raw = df_new["pinnacle_ml_home_prob"].astype(float)
            pa_raw = df_new["pinnacle_ml_away_prob"].astype(float)
            denom = ph_raw + pa_raw
            df_new["pinnacle_fair_home_prob"] = ph_raw / denom
            df_new["pinnacle_fair_away_prob"] = pa_raw / denom
        else:
            raise ValueError("Need Pinnacle fair probabilities or raw home/away probs.")

    # Extract X for prediction
    X_new = df_new[feature_cols].copy()
    X_new = X_new.apply(lambda col: col.fillna(col.median()))

    # Model probabilities
    df_new["p_home_logit_calib"] = calib_model.predict_proba(X_new)[:, 1]
    df_new["p_home_xgb"] = xgb_model.predict_proba(X_new)[:, 1]

    # Sides
    df_new["logit_side"] = np.where(df_new["p_home_logit_calib"] > 0.5, "home", "away")
    df_new["xgb_side"] = np.where(df_new["p_home_xgb"] > 0.5, "home", "away")

    df_new["consensus_side"] = np.where(
        df_new["logit_side"] == df_new["xgb_side"],
        df_new["logit_side"],
        "none"
    )

    # Edges vs Pinnacle fair
    df_new["edge_home_calib"] = df_new["p_home_logit_calib"] - df_new["pinnacle_fair_home_prob"]
    df_new["edge_away_calib"] = (1 - df_new["p_home_logit_calib"]) - df_new["pinnacle_fair_away_prob"]

    df_new["edge_home_xgb"] = df_new["p_home_xgb"] - df_new["pinnacle_fair_home_prob"]
    df_new["edge_away_xgb"] = (1 - df_new["p_home_xgb"]) - df_new["pinnacle_fair_away_prob"]

    # Apply consensus rule for upcoming games
    df_new["bet_side"] = "none"

    # HOME
    home_mask = df_new["consensus_side"].eq("home")
    if require_logit_edge_positive:
        home_mask = home_mask & (df_new["edge_home_calib"] > 0)
    home_mask = home_mask & (df_new["edge_home_xgb"] >= edge_threshold)
    df_new.loc[home_mask, "bet_side"] = "home"

    # AWAY
    away_mask = df_new["consensus_side"].eq("away")
    if require_logit_edge_positive:
        away_mask = away_mask & (df_new["edge_away_calib"] > 0)
    away_mask = away_mask & (df_new["edge_away_xgb"] >= edge_threshold)
    df_new.loc[away_mask, "bet_side"] = "away"

    bets_today = df_new[df_new["bet_side"] != "none"].copy()
    if bets_today.empty:
        print("No consensus bets today with current threshold.")
        return bets_today

    # Attach bet price
    bets_today["bet_price_american"] = np.where(
        bets_today["bet_side"] == "home",
        bets_today["pinnacle_ml_home"],
        bets_today["pinnacle_ml_away"]
    )

    # Model prob from XGB for stake sizing
    bets_today["p_model"] = np.where(
        bets_today["bet_side"] == "home",
        bets_today["p_home_xgb"],
        1.0 - bets_today["p_home_xgb"]
    )

    # Kelly stake sizing
    bets_today = add_kelly_columns(bets_today)

    # Compute final recommended stake per bet
    stakes = []
    for _, row in bets_today.iterrows():
        k_full = max(row["kelly_full"], 0.0)
        stake_fraction = min(frac_kelly * k_full, max_fraction_per_bet)
        stake_units = stake_fraction * bankroll
        stakes.append(stake_units)

    bets_today["stake_units"] = stakes

    # Pick key columns for your bet sheet
    cols_out = [
        "game_id",
        "game_date",
        "home_team",
        "away_team",
        "bet_side",
        "bet_price_american",
        "p_home_xgb",
        "p_home_logit_calib",
        "pinnacle_fair_home_prob",
        "pinnacle_fair_away_prob",
        "edge_home_xgb",
        "edge_away_xgb",
        "kelly_full",
        "stake_units"
    ]

    cols_out = [c for c in cols_out if c in bets_today.columns]
    bets_today = bets_today[cols_out].sort_values("game_date")

    return bets_today

# NOTE:
# To use generate_daily_bets, you will need:
# - games_features_df for upcoming games with the same features + game_id, home_team, away_team
# - pinn_odds_df with Pinnacle odds and fair probs, keyed by game_id
# - the fitted 'log_reg_calibrated' and 'xgb_model' objects
# - the list 'feature_cols' you used earlier

# Example (pseudo-usage):
# bets_today = generate_daily_bets(
#     games_features_df=upcoming_games_df,
#     pinn_odds_df=upcoming_pinn_df,
#     calib_model=log_reg_calibrated,
#     xgb_model=xgb_model,
#     feature_cols=feature_cols,
#     edge_threshold=0.02,
#     frac_kelly=0.25,
#     max_fraction_per_bet=0.05,
#     bankroll=1000.0
# )
# bets_today.to_csv("bets_today.csv", index=False)


Consensus VAL bets: 272
Consensus TEST bets: 275

=== Kelly Simulation (TEST Consensus) ===
initial_bankroll: 1000.0
final_bankroll: 1436.738643164457
total_profit: 436.738643164457
roi_total: 0.43673864316445704
num_bets: 274
hit_rate: 0.5328467153284672
max_drawdown: 0.6192386152905451
frac_kelly: 0.25
max_fraction_per_bet: 0.05

=== Line Range Analysis (VAL, Consensus) ===
             price_band  count  total_profit  roi_per_bet
0  (-10000.001, -300.0]      6      0.043010     0.007168
1      (-300.0, -150.0]     50      5.297826     0.105957
2         (-150.0, 0.0]    101      1.652005     0.016356
3          (0.0, 150.0]     56    -13.820000    -0.246786
4        (150.0, 300.0]     46      1.710000     0.037174
5      (300.0, 10000.0]     13      8.390000     0.645385

=== Line Range Analysis (TEST, Consensus) ===
             price_band  count  total_profit  roi_per_bet
0  (-10000.001, -300.0]      4      0.922025     0.230506
1      (-300.0, -150.0]     45      7.211563     0.1

/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_32035/3250596649.py:303: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = bets.groupby("price_band")["profit_flat_unit"].agg(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_32035/3250596649.py:303: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = bets.groupby("price_band")["profit_flat_unit"].agg(
